# 🏛️ 梵天-mini 训练脚本
## 基于《动手学大模型》Chapter 4 改造 | 苏摩111封印 2026-09-05

**目标：** 用梵天历史信号数据对 Qwen2.5-1.5B 做 LoRA 微调

**改造点：**
- 原版：DeepMath-103K 数学推理数据集
- 梵天版：brahma_sft_train.jsonl 信号裁决数据集
- 原版：Qwen2.5-Math-1.5B
- 梵天版：Qwen2.5-1.5B-Instruct（中文更强）
- 降精度：40GB显存 → 16GB T4 用 4bit 量化

**预计时间：** T4 免费 GPU 约 2-4 小时

## Step 0: 检查 GPU 环境

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无GPU"}')
print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB' if torch.cuda.is_available() else '')
print(f'CUDA: {torch.version.cuda}')
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## Step 1: 安装依赖

In [ ]:
!pip install -q transformers==4.44.0 peft==0.12.0 trl==0.10.1 \
    bitsandbytes==0.43.3 accelerate==0.33.0 datasets==2.21.0 \
    sentencepiece protobuf einops

## Step 2: 上传梵天训练数据

In [ ]:
# 方式A: 从Google Drive加载（推荐）
from google.colab import drive
drive.mount('/content/drive')

# 把 brahma_sft_train.jsonl 上传到 Google Drive 后修改路径
DATA_PATH = '/content/drive/MyDrive/brahma_sft_train.jsonl'

# 验证数据
import json
samples = [json.loads(l) for l in open(DATA_PATH) if l.strip()]
print(f'训练样本总数: {len(samples)}')
print(f'\n样本预览:')
for s in samples[:2]:
    print(f'  instruction: {s["instruction"][:60]}')
    print(f'  output:      {s["output"][:60]}')
    print()

## Step 3: 加载基座模型（4bit量化，适配16GB T4）

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

# 4bit量化配置（关键：把40GB显存需求降到4GB）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f'加载模型: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side='right',
)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

print(f'模型加载完成')
print(f'参数量: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')

## Step 4: 配置 LoRA（梵天专属参数）

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 准备4bit训练
model = prepare_model_for_kbit_training(model)

# LoRA配置（梵天信号裁决任务专属）
lora_config = LoraConfig(
    r=16,                          # rank，16适合小数据集
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 预期输出：trainable params: ~2M (约0.1%总参数)

## Step 5: 构建梵天训练数据集

In [ ]:
from datasets import Dataset

# 梵天宪法System Prompt（注入每条训练样本）
BRAHMA_SYSTEM = """你是梵天量化系统的信号裁决员。
梵天宪法铁律（不可违反）：
1. BEAR_TREND体制做多WR=45% → 必须输出AVOID
2. BULL_TREND体制做空WR=38% → 必须输出AVOID
3. SL距离必须>=1.5×ATR1H → 不满足输出WAIT
4. 无FVG+OB+清算三因子共振 → 输出WAIT不给入场价
5. 回答格式：ACTION:LONG|SHORT|WAIT|AVOID  CONF:HIGH|MED|LOW  REASON:一句话"""

def format_sample(sample):
    """转成Qwen2.5的对话格式"""
    messages = [
        {'role': 'system',    'content': BRAHMA_SYSTEM},
        {'role': 'user',      'content': sample['instruction'] + ('\n' + sample['input'] if sample.get('input') else '')},
        {'role': 'assistant', 'content': sample['output']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': text}

# 构建数据集
raw_data = [json.loads(l) for l in open(DATA_PATH) if l.strip()]
formatted = [format_sample(s) for s in raw_data]
dataset = Dataset.from_list(formatted)

# 按8:2划分训练/验证集
split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'训练集: {len(train_dataset)} 条')
print(f'验证集: {len(eval_dataset)} 条')
print(f'\n样本预览（前100字符）:')
print(train_dataset[0]['text'][:200])

## Step 6: 训练配置 + 开始训练

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

training_args = SFTConfig(
    output_dir='/content/brahma_mini_checkpoints',
    num_train_epochs=3,              # 400条数据跑3轮
    per_device_train_batch_size=2,   # T4 16GB，batch=2
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # 等效batch=16
    learning_rate=2e-4,
    fp16=True,                       # T4用fp16
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    report_to='none',                # 不用wandb
    max_seq_length=512,              # 梵天信号文本不超512
    dataset_text_field='text',
    optim='paged_adamw_8bit',        # 8bit优化器节省显存
    gradient_checkpointing=True,     # 进一步节省显存
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

print('开始训练梵天-mini...')
print(f'预计时间: T4 约 2-4 小时')
trainer.train()

## Step 7: 保存模型

In [ ]:
# 保存LoRA权重
SAVE_PATH = '/content/drive/MyDrive/brahma_mini_lora'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'LoRA权重保存到: {SAVE_PATH}')

# 合并LoRA到基座模型（可选，推理更快）
from peft import PeftModel
merged_model = model.merge_and_unload()
MERGED_PATH = '/content/drive/MyDrive/brahma_mini_merged'
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)
print(f'合并模型保存到: {MERGED_PATH}')

## Step 8: 梵天宪法测试（验证训练效果）

In [ ]:
model.eval()

# 测试用例：梵天宪法铁律验证
test_cases = [
    # 铁律1：BEAR_TREND做多必须AVOID
    {'instruction': 'BTC/USDT BEAR_TREND体制 score=145 是否做多？',
     'expected': 'AVOID'},
    # 铁律2：BULL_TREND做空必须AVOID
    {'instruction': 'ETH/USDT BULL_TREND体制 score=152 是否做空？',
     'expected': 'AVOID'},
    # 正常信号：BEAR_TREND做空应该ENTER
    {'instruction': 'BTC/USDT BEAR_TREND体制 score=148 FVG向下 OI做空 是否做空？',
     'expected': 'ENTER 或 SHORT'},
    # CHOP体制：应该WAIT
    {'instruction': 'ETH/USDT CHOP_MID体制 score=112 是否入场？',
     'expected': 'WAIT'},
]

print('=== 梵天宪法铁律验证 ===')
for i, tc in enumerate(test_cases):
    messages = [
        {'role': 'system', 'content': BRAHMA_SYSTEM},
        {'role': 'user',   'content': tc['instruction']},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.1, do_sample=False)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    ok = tc['expected'].split()[0] in response.upper()
    print(f'[{"✅" if ok else "❌"}] 测试{i+1}: {tc["instruction"][:40]}')
    print(f'     期望: {tc["expected"]}  实际: {response[:60]}')
    print()

## Step 9（可选）: 量化为 GGUF 部署到 james-bond

训练完成后，把模型量化为 GGUF 格式，部署到 james-bond 本地推理（< 1秒）

```bash
# james-bond 上执行
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp && make -j4

# 转换为GGUF
python3 convert_hf_to_gguf.py ~/brahma_mini_merged --outfile brahma_mini_q4.gguf --outtype q4_k_m

# 测试推理
./llama-cli -m brahma_mini_q4.gguf -p 'BTC BEAR_TREND score=145 做多还是做空？' -n 50
```